In [36]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase
from pathlib import Path

In [37]:
env_path = Path("/home/luis/Documents/FGV/Laboratory/document-graph/server/.env")

load_dotenv(dotenv_path=env_path)

neo4j_uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
neo4j_username = os.getenv("NEO4J_USERNAME", "neo4j")
neo4j_password = os.getenv("NEO4J_PASSWORD", "neo4j_password")
neo4j_database = os.getenv("NEO4J_DATABASE", "neo4j")

In [38]:
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(neo4j_username, neo4j_password),
)

def run_query(query: str, params: dict | None = None):
    with driver.session(database=neo4j_database) as session:
        result = session.run(query, params or {})
        return [r.data() for r in result]


In [39]:
# 1) Ver contratos cargados
run_query("""
MATCH (c:Contract)
RETURN c.path AS path, c.content_hash AS content_hash
ORDER BY path
""")


[{'path': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.json',
  'content_hash': '4a2187cb60da96c331433608de54893734fbde45'},
 {'path': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/BLUEFLYINC_03_27_2002-EX-10.27-e-business Hosting Agreement.json',
  'content_hash': 'cd18362093e0c4ba3cbcb68912fd0ef5459d35bf'},
 {'path': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/HealthcentralCom_19991108_S-1A_EX-10.27_6623292_EX-10.27_Co-Branding Agreement.json',
  'content_hash': '7fd9202dec18af0ff98136de4ba2c2998f4c1a13'},
 {'path': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement.json',
  'content_hash': 'f3245eb60b42b2e3c7f79eb0f075801474bdcb46'}]

In [40]:
# 2) Conteo por tipo de nodo
run_query("""
MATCH (n:KGNode)
RETURN n.node_type AS node_type, count(*) AS total
ORDER BY total DESC
""")


[{'node_type': 'Clause', 'total': 921},
 {'node_type': 'Obligation', 'total': 84},
 {'node_type': 'DefinedTerm', 'total': 38},
 {'node_type': 'Right', 'total': 26},
 {'node_type': 'Prohibition', 'total': 16},
 {'node_type': 'Party', 'total': 10},
 {'node_type': 'Condition', 'total': 9},
 {'node_type': 'Reference', 'total': 2},
 {'node_type': 'Value', 'total': 2}]

In [41]:
# 3) Conteo por tipo de relación
run_query("""
MATCH (:KGNode)-[r]->(:KGNode)
RETURN type(r) AS rel_type, count(*) AS total
ORDER BY total DESC
""")


[{'rel_type': 'REFERENCES', 'total': 659},
 {'rel_type': 'CONTAINS', 'total': 239},
 {'rel_type': 'IS_PART_OF', 'total': 233},
 {'rel_type': 'ASSIGNS_OBLIGATION_TO', 'total': 70},
 {'rel_type': 'GRANTS_RIGHT_TO', 'total': 30},
 {'rel_type': 'DEFINES', 'total': 10},
 {'rel_type': 'DEPENDS_ON', 'total': 9},
 {'rel_type': 'USES', 'total': 6}]

In [42]:
run_query("""
MATCH (ct:Contract)
OPTIONAL MATCH (ct)-[:CONTAINS]->(n:KGNode)
WITH ct, count(DISTINCT n) AS nodes
OPTIONAL MATCH (ct)-[:CONTAINS]->(a:KGNode)-[r]-(:KGNode)<-[:CONTAINS]-(ct)
RETURN
  ct.path AS contract,
  nodes,
  count(DISTINCT id(r)) AS edges
ORDER BY contract
""")


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=9, column=18, offset=233>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 233, 'line': 9, 'column': 18}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (ct:Contract)\nOPTIONAL MATCH (ct)-[:CONTAINS]->(n:KGNode)\nWITH ct, count(DISTINCT n) AS nodes\nOPTIONAL MATCH (ct)-[:CONTAINS]->(a:KGNode)-[r]-(:KGNode)<-[:CONTAINS]-(ct)\nRETURN\n  ct.path AS contract,\n  nodes,\n  count(DISTINCT id(r)) AS edges\nORDER BY contract\n'


[{'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.json',
  'nodes': 490,
  'edges': 775},
 {'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/BLUEFLYINC_03_27_2002-EX-10.27-e-business Hosting Agreement.json',
  'nodes': 265,
  'edges': 233},
 {'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/HealthcentralCom_19991108_S-1A_EX-10.27_6623292_EX-10.27_Co-Branding Agreement.json',
  'nodes': 238,
  'edges': 97},
 {'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement.json',
  'nodes': 115,
  'edges': 151}]

In [43]:
run_query("""
MATCH (ct:Contract)
OPTIONAL MATCH (ct)-[:CONTAINS]->(n:KGNode)
WITH ct, collect(DISTINCT n) AS nodes
UNWIND nodes AS n
OPTIONAL MATCH (ct)-[:CONTAINS]->(n)-[r]-(:KGNode)<-[:CONTAINS]-(ct)
WITH ct.path AS contract, n, count(DISTINCT id(r)) AS deg
WITH
  contract,
  count(n) AS total_nodes,
  sum(CASE WHEN deg > 0 THEN 1 ELSE 0 END) AS connected_nodes,
  sum(CASE WHEN deg = 0 THEN 1 ELSE 0 END) AS isolated_nodes,
  round(avg(deg), 3) AS avg_degree
RETURN
  contract,
  total_nodes,
  connected_nodes,
  isolated_nodes,
  round(100.0 * isolated_nodes / total_nodes, 2) AS isolated_pct,
  avg_degree
ORDER BY isolated_pct DESC, contract
""")


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=7, column=45, offset=234>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 234, 'line': 7, 'column': 45}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (ct:Contract)\nOPTIONAL MATCH (ct)-[:CONTAINS]->(n:KGNode)\nWITH ct, collect(DISTINCT n) AS nodes\nUNWIND nodes AS n\nOPTIONAL MATCH (ct)-[:CONTAINS]->(n)-[r]-(:KGNode)<-[:CONTAINS]-(ct)\nWITH ct.path AS contract, n, count(DISTINCT id(r)) AS deg\nWITH\n  contract,\n  count(n) AS total_nodes,\n  sum(CASE WHEN deg > 0 THEN 1 ELSE 0 EN

[{'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/HealthcentralCom_19991108_S-1A_EX-10.27_6623292_EX-10.27_Co-Branding Agreement.json',
  'total_nodes': 238,
  'connected_nodes': 65,
  'isolated_nodes': 173,
  'isolated_pct': 72.69,
  'avg_degree': 0.815},
 {'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/BLUEFLYINC_03_27_2002-EX-10.27-e-business Hosting Agreement.json',
  'total_nodes': 265,
  'connected_nodes': 152,
  'isolated_nodes': 113,
  'isolated_pct': 42.64,
  'avg_degree': 1.758},
 {'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement.json',
  'total_nodes': 115,
  'connected_nodes': 75,
  'isolated_nodes': 40,
  'isolated_pct': 34.78,
  'avg_degree': 2.626},
 {'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.json',
  'total

In [44]:
run_query("""
MATCH (ct:Contract)-[:CONTAINS]->(n:KGNode)
OPTIONAL MATCH (ct)-[:CONTAINS]->(n)-[r]-(:KGNode)<-[:CONTAINS]-(ct)
WITH ct.path AS contract, n, count(DISTINCT id(r)) AS deg
WHERE deg = 0
RETURN
  contract,
  n.node_type AS isolated_node_type,
  count(*) AS total
ORDER BY contract, total DESC
""")


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=45, offset=158>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 158, 'line': 4, 'column': 45}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (ct:Contract)-[:CONTAINS]->(n:KGNode)\nOPTIONAL MATCH (ct)-[:CONTAINS]->(n)-[r]-(:KGNode)<-[:CONTAINS]-(ct)\nWITH ct.path AS contract, n, count(DISTINCT id(r)) AS deg\nWHERE deg = 0\nRETURN\n  contract,\n  n.node_type AS isolated_node_type,\n  count(*) AS total\nORDER BY contract, total DESC\n'


[{'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.json',
  'isolated_node_type': 'Clause',
  'total': 98},
 {'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/BLUEFLYINC_03_27_2002-EX-10.27-e-business Hosting Agreement.json',
  'isolated_node_type': 'Clause',
  'total': 113},
 {'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/HealthcentralCom_19991108_S-1A_EX-10.27_6623292_EX-10.27_Co-Branding Agreement.json',
  'isolated_node_type': 'Clause',
  'total': 173},
 {'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement.json',
  'isolated_node_type': 'Clause',
  'total': 40}]

In [45]:
# run_query("""
# MATCH (ct:Contract)-[:CONTAINS]->(c:Clause)
# MATCH (ct)-[:CONTAINS]->(c)-[r]-(m:KGNode)<-[:CONTAINS]-(ct)
# RETURN
#   ct.path AS contract,
#   c.node_id AS clause_id,
#   m.node_type AS neighbor_type,
#   count(DISTINCT id(r)) AS rel_count
# ORDER BY contract, clause_id, rel_count DESC
# """)


In [46]:
run_query("""
MATCH (ct:Contract)-[:CONTAINS]->(c:Clause {paragraph_uid: $paragraph_uid})
OPTIONAL MATCH (ct)-[:CONTAINS]->(c)-[r]-(m:KGNode)<-[:CONTAINS]-(ct)
WITH ct, c,
     collect(DISTINCT CASE
       WHEN r IS NULL THEN NULL
       ELSE {
         relation: type(r),
         direction: CASE WHEN startNode(r)=c THEN "OUT" ELSE "IN" END,
         node_id: m.node_id,
         node_type: m.node_type,
         label: m.label
       }
     END) AS raw_neighbors
WITH ct, c, [x IN raw_neighbors WHERE x IS NOT NULL] AS neighbors
RETURN
  ct.path AS contract,
  c.node_id AS clause_node_id,
  c.source_paragraph_id AS source_paragraph_id,
  c.paragraph_uid AS paragraph_uid,
  size(neighbors) = 0 AS is_isolated,
  neighbors
ORDER BY contract;
""", params={"paragraph_uid": "1152e6cb7388c71fdafe8f45698bd0200f7e438e"})

[{'contract': '/home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.json',
  'clause_node_id': '4a2187cb60da96c331433608de54893734fbde45:Clause:BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement-p-10',
  'source_paragraph_id': 'BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement-p-10',
  'paragraph_uid': '1152e6cb7388c71fdafe8f45698bd0200f7e438e',
  'is_isolated': False,
  'neighbors': [{'node_type': 'Party',
    'label': 'Miltenyi Biotec GmbH',
    'relation': 'CONTAINS',
    'node_id': '4a2187cb60da96c331433608de54893734fbde45:Party:bcdc9ac6edf65122',
    'direction': 'OUT'},
   {'node_type': 'Party',
    'label': 'Bellicum Pharmaceuticals, Inc',
    'relation': 'CONTAINS',
    'node_id': '4a2187cb60da96c331433608de54893734fbde45:Party:4e08f283873a4ed2',
    'direction': 'OUT'},
   {'node_type': 'Party',
    'label': 'Miltenyi Biotec GmbH',
    'relation': 'IS_PART_OF',
    'node_i

In [47]:
document_list = [
    "BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement",
    "BLUEFLYINC_03_27_2002-EX-10.27-e-business Hosting Agreement",
    "HealthcentralCom_19991108_S-1A_EX-10.27_6623292_EX-10.27_Co-Branding Agreement",
    "LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement",
]

In [48]:
import json

MAX = 5

docs = {}
for js in document_list:
    relationsCounts = []
    # with open(f"../../infra/json/graph/{js}.json", "r") as f:
    #     data = json.load(f)
    with open(f"../../infra/json/graph/{js}.json", "r") as f:
        data = json.load(f)

    nodes = data.get("nodes", [])
    edges = data.get("edges", [])

    for node in nodes:
        node_id = node.get("id")
        paragraph_uid = node.get("paragraph_uid")
        relationsCount = node.get("relationsCount", 0)
        relationsCounts.append((node_id, relationsCount))
    
    relationsCounts.sort(key=lambda x: x[1], reverse=True)
    top_relations = relationsCounts[:MAX]
    
    docs[js] = top_relations

In [49]:
def analyze_clause_neighbors(paragraph_uid: str):
    return run_query("""
    MATCH (ct:Contract)-[:CONTAINS]->(c:Clause {paragraph_uid: $paragraph_uid})
    OPTIONAL MATCH (ct)-[:CONTAINS]->(c)-[r]-(m:KGNode)<-[:CONTAINS]-(ct)
    WITH ct, c,
        collect(DISTINCT CASE
        WHEN r IS NULL THEN NULL
        ELSE {
            relation: type(r),
            direction: CASE WHEN startNode(r)=c THEN "OUT" ELSE "IN" END,
            node_id: m.node_id,
            node_type: m.node_type,
            label: m.label
        }
        END) AS raw_neighbors
    WITH ct, c, [x IN raw_neighbors WHERE x IS NOT NULL] AS neighbors
    RETURN
    ct.path AS contract,
    c.node_id AS clause_node_id,
    c.source_paragraph_id AS source_paragraph_id,
    c.paragraph_uid AS paragraph_uid,
    size(neighbors) = 0 AS is_isolated,
    neighbors
    ORDER BY contract;
    """, params={"paragraph_uid": f"{paragraph_uid}"})

In [50]:
# for key, value in docs.items():
#     for node_id, relationsCount in value:
#         print(f"Document: {key}, Node ID: {node_id}, Relations Count: {relationsCount}")
#         neighbors_info = analyze_clause_neighbors(node_id)
#         print(json.dumps(neighbors_info, indent=2))

In [51]:
import json
import pandas as pd
from pathlib import Path

q = """
MATCH (ct:Contract)-[:CONTAINS]->(c:Clause)
OPTIONAL MATCH (ct)-[:CONTAINS]->(c)-[r]-(:KGNode)<-[:CONTAINS]-(ct)
WITH ct.path AS contract, c, count(DISTINCT id(r)) AS kg_degree
WHERE kg_degree > 0
RETURN
  contract,
  c.node_id AS clause_node_id,
  c.source_paragraph_id AS source_paragraph_id,
  c.paragraph_uid AS paragraph_uid,
  kg_degree
"""

def get_relcount(contract_path: str, source_paragraph_id: str) -> int:
    if contract_path not in rel_:
        p = Path(contract_path)
        payload = json.loads(p.read_text(encoding="utf-8"))
        if isinstance(payload.get("graph"), dict):
            payload = payload["graph"]
        rel_map = {}
        for n in payload.get("nodes", []):
            nid = str(n.get("id", "")).strip()
            if nid:
                rel_map[nid] = int(n.get("relationsCount", 0) or 0)
        rel_[contract_path] = rel_map
    return rel_[contract_path].get(str(source_paragraph_id), 0)

kg_df = pd.DataFrame(run_query(q))

if kg_df.empty:
    print("No hay cláusulas no aisladas en KG.")
else:
    rel_ = {}
    kg_df["base_relationsCount"] = kg_df.apply(
        lambda x: get_relcount(x["contract"], x["source_paragraph_id"]), axis=1
    )

    top5 = (
        kg_df.sort_values(
            by=["contract", "base_relationsCount", "kg_degree"],
            ascending=[True, False, False],
        )
        .groupby("contract", as_index=False)
        .head(5)
        .reset_index(drop=True)
    )

    display(
        top5[
            ["contract", "source_paragraph_id", "paragraph_uid", "base_relationsCount", "kg_degree"]
        ]
    )


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=45, offset=158>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 158, 'line': 4, 'column': 45}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (ct:Contract)-[:CONTAINS]->(c:Clause)\nOPTIONAL MATCH (ct)-[:CONTAINS]->(c)-[r]-(:KGNode)<-[:CONTAINS]-(ct)\nWITH ct.path AS contract, c, count(DISTINCT id(r)) AS kg_degree\nWHERE kg_degree > 0\nRETURN\n  contract,\n  c.node_id AS clause_node_id,\n  c.source_paragraph_id AS source_paragraph_id,\n  c.paragraph_uid AS paragraph_uid,\n

,contract,source_paragraph_id,paragraph_uid,base_relationsCount,kg_degree
0,/home/luis/Documents/FGV/Laboratory/document-g...,"BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1...",6d843a956f1adcbf7394d27457e8342f020a9ed8,121,121
1,/home/luis/Documents/FGV/Laboratory/document-g...,"BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1...",c3dcf35efbbd129ee4e08d34cb26109be2dc9737,36,31
2,/home/luis/Documents/FGV/Laboratory/document-g...,"BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1...",102fafa0ede1ad48835bc9d844bf234a120a5986,35,31
3,/home/luis/Documents/FGV/Laboratory/document-g...,"BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1...",27aa2a37e94374ded441a3fc6491ab467ffa30d1,34,31
4,/home/luis/Documents/FGV/Laboratory/document-g...,"BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1...",c0ca305f958217b50991bc9690c0c2e5abb4a281,27,27
5,/home/luis/Documents/FGV/Laboratory/document-g...,BLUEFLYINC_03_27_2002-EX-10.27-e-business Host...,5b869cb5dd4181e2d1fa9369658bade12df646aa,23,23
6,/home/luis/Documents/FGV/Laboratory/document-g...,BLUEFLYINC_03_27_2002-EX-10.27-e-business Host...,73d6b2e02e341ef55b895737ad0340470d984fb0,8,8
7,/home/luis/Documents/FGV/Laboratory/document-g...,BLUEFLYINC_03_27_2002-EX-10.27-e-business Host...,c222917d975e61e910f6bd65ca4d075e445d9692,8,8
8,/home/luis/Documents/FGV/Laboratory/document-g...,BLUEFLYINC_03_27_2002-EX-10.27-e-business Host...,e0633e8c9bba5eb0a6944c50bb5eb38b71b7b184,6,6
9,/home/luis/Documents/FGV/Laboratory/document-g...,BLUEFLYINC_03_27_2002-EX-10.27-e-business Host...,e66ef6eee17776881f3fa30b0de56fd46c847b2b,6,5


In [52]:
# driver.close()


In [53]:
import json
import pandas as pd
from pathlib import Path


def select_top_k_non_isolated(document_list: list[str], top_k: int = 5) -> pd.DataFrame:
    q = """
    MATCH (ct:Contract)-[:CONTAINS]->(c:Clause)
    OPTIONAL MATCH (ct)-[:CONTAINS]->(c)-[r]-(:KGNode)<-[:CONTAINS]-(ct)
    WITH ct.path AS contract, c, count(DISTINCT id(r)) AS kg_degree
    WHERE kg_degree > 0
      AND ANY(doc IN $document_list WHERE contract CONTAINS doc)
    RETURN
      contract,
      c.node_id AS clause_node_id,
      c.source_paragraph_id AS source_paragraph_id,
      c.paragraph_uid AS paragraph_uid,
      kg_degree
    """

    kg_df = pd.DataFrame(run_query(q, params={"document_list": document_list}))
    if kg_df.empty:
        return kg_df

    rel_cache: dict[str, dict[str, int]] = {}

    def get_relcount(contract_path: str, source_paragraph_id: str) -> int:
        if contract_path not in rel_cache:
            payload = json.loads(Path(contract_path).read_text(encoding="utf-8"))
            if isinstance(payload.get("graph"), dict):
                payload = payload["graph"]
            rel_map = {}
            for n in payload.get("nodes", []):
                nid = str(n.get("id", "")).strip()
                if nid:
                    rel_map[nid] = int(n.get("relationsCount", 0) or 0)
            rel_cache[contract_path] = rel_map
        return rel_cache[contract_path].get(str(source_paragraph_id), 0)

    kg_df["base_relationsCount"] = kg_df.apply(
        lambda x: get_relcount(x["contract"], x["source_paragraph_id"]), axis=1
    )

    topk = (
        kg_df.sort_values(
            by=["contract", "base_relationsCount", "kg_degree"],
            ascending=[True, False, False],
        )
        .groupby("contract", as_index=False)
        .head(top_k)
        .reset_index(drop=True)
    )
    return topk


def _fetch_seed_clause(paragraph_uid: str) -> dict:
    q = """
    MATCH (ct:Contract)-[:CONTAINS]->(c:Clause {paragraph_uid: $paragraph_uid})
    RETURN
      ct.path AS contract,
      c.node_id AS clause_node_id,
      c.source_paragraph_id AS source_paragraph_id,
      c.paragraph_uid AS paragraph_uid,
      coalesce(c.text, "") AS clause_text
    LIMIT 1
    """
    rows = run_query(q, params={"paragraph_uid": paragraph_uid})
    return rows[0] if rows else {}


def _fetch_hop_edges(paragraph_uid: str, hops: int) -> list[dict]:
    if hops not in (1, 2):
        raise ValueError("hops must be 1 or 2")

    q = f"""
    MATCH (ct:Contract)-[:CONTAINS]->(c:Clause {{paragraph_uid: $paragraph_uid}})
    MATCH p=(c)-[*1..{hops}]-(n:KGNode)
    WHERE ALL(x IN nodes(p)[1..] WHERE (ct)-[:CONTAINS]->(x))
    UNWIND relationships(p) AS rel
    WITH DISTINCT ct, rel, startNode(rel) AS s, endNode(rel) AS t
    RETURN
      ct.path AS contract,
      s.node_id AS source_id,
      s.node_type AS source_type,
      s.label AS source_label,
      properties(s) AS source_props,
      type(rel) AS rel_type,
      properties(rel) AS rel_props,
      t.node_id AS target_id,
      t.node_type AS target_type,
      t.label AS target_label,
      properties(t) AS target_props
    ORDER BY rel_type, source_id, target_id
    """
    return run_query(q, params={"paragraph_uid": paragraph_uid})


def _node_snippet(node_type: str, props: dict) -> str:
    keys = ["text", "term", "name", "action", "trigger", "amount", "definition", "citation"]
    text = ""
    for k in keys:
        val = props.get(k)
        if isinstance(val, str) and val.strip():
            text = val.strip()
            break
    if not text:
        text = props.get("id_raw", "") or ""
    text = str(text).replace("\n", " ").strip()
    if len(text) > 220:
        text = text[:220] + "..."
    return f"{node_type}: {text}" if text else node_type


def build_prompt_context_for_clause(paragraph_uid: str, hops: int = 1) -> dict:
    seed = _fetch_seed_clause(paragraph_uid)
    if not seed:
        return {
            "paragraph_uid": paragraph_uid,
            "is_isolated": True,
            "nodes": [],
            "relations": [],
            "prompt_context": "[SEED]\nnot_found"
        }

    rows = _fetch_hop_edges(paragraph_uid, hops=hops)
    node_map = {}
    rel_set = set()

    # include seed node in node_map
    node_map[seed["clause_node_id"]] = {
        "node_id": seed["clause_node_id"],
        "node_type": "Clause",
        "label": seed.get("source_paragraph_id") or seed["clause_node_id"],
        "props": {
            "text": seed.get("clause_text", ""),
            "paragraph_uid": seed.get("paragraph_uid"),
            "source_paragraph_id": seed.get("source_paragraph_id"),
        },
    }

    for r in rows:
        s_id = r["source_id"]
        t_id = r["target_id"]

        node_map[s_id] = {
            "node_id": s_id,
            "node_type": r.get("source_type", "KGNode"),
            "label": r.get("source_label", ""),
            "props": r.get("source_props") or {},
        }
        node_map[t_id] = {
            "node_id": t_id,
            "node_type": r.get("target_type", "KGNode"),
            "label": r.get("target_label", ""),
            "props": r.get("target_props") or {},
        }

        rel_key = (s_id, r["rel_type"], t_id)
        rel_set.add(rel_key)

    nodes = list(node_map.values())
    relations = [{"source_id": s, "rel_type": rt, "target_id": t} for (s, rt, t) in sorted(rel_set)]
    is_isolated = len(relations) == 0

    # prompt-friendly text
    lines = []
    lines.append("[SEED]")
    lines.append(f"contract: {seed['contract']}")
    lines.append(f"clause_node_id: {seed['clause_node_id']}")
    lines.append(f"source_paragraph_id: {seed.get('source_paragraph_id')}")
    lines.append(f"paragraph_uid: {seed.get('paragraph_uid')}")
    lines.append(f"seed_text: {str(seed.get('clause_text','')).replace(chr(10), ' ').strip()[:500]}")
    lines.append("")

    lines.append(f"[NODES_{hops}HOP]")
    for n in sorted(nodes, key=lambda x: (x.get("node_type", ""), x.get("node_id", ""))):
        snippet = _node_snippet(n.get("node_type", "KGNode"), n.get("props") or {})
        lines.append(f"- {n['node_id']} | {n.get('node_type')} | {snippet}")
    lines.append("")

    lines.append(f"[RELATIONS_{hops}HOP]")
    for rel in relations:
        lines.append(f"- {rel['source_id']} -[{rel['rel_type']}]-> {rel['target_id']}")

    return {
        "paragraph_uid": paragraph_uid,
        "contract": seed["contract"],
        "clause_node_id": seed["clause_node_id"],
        "source_paragraph_id": seed.get("source_paragraph_id"),
        "is_isolated": is_isolated,
        "nodes": nodes,
        "relations": relations,
        "prompt_context": "\n".join(lines),
    }


def build_1hop_prompt_contexts(document_list: list[str], top_k: int = 5) -> list[dict]:
    topk_df = select_top_k_non_isolated(document_list=document_list, top_k=top_k)
    out = []
    for row in topk_df.to_dict("records"):
        out.append(build_prompt_context_for_clause(row["paragraph_uid"], hops=1))
    return out


def build_2hop_prompt_contexts(document_list: list[str], top_k: int = 5) -> list[dict]:
    topk_df = select_top_k_non_isolated(document_list=document_list, top_k=top_k)
    out = []
    for row in topk_df.to_dict("records"):
        out.append(build_prompt_context_for_clause(row["paragraph_uid"], hops=2))
    return out


# Example usage:
topk_df = select_top_k_non_isolated(document_list, top_k=5)
display(topk_df[["contract", "source_paragraph_id", "paragraph_uid", "base_relationsCount", "kg_degree"]])
one_hop_payloads = build_1hop_prompt_contexts(document_list, top_k=5)
two_hop_payloads = build_2hop_prompt_contexts(document_list, top_k=5)
print(two_hop_payloads[0]["prompt_context"])



Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=49, offset=170>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 170, 'line': 4, 'column': 49}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (ct:Contract)-[:CONTAINS]->(c:Clause)\n    OPTIONAL MATCH (ct)-[:CONTAINS]->(c)-[r]-(:KGNode)<-[:CONTAINS]-(ct)\n    WITH ct.path AS contract, c, count(DISTINCT id(r)) AS kg_degree\n    WHERE kg_degree > 0\n      AND ANY(doc IN $document_list WHERE contract CONTAINS doc)\n    RETURN\n      contract,\n      c.node_id AS clause_no

,contract,source_paragraph_id,paragraph_uid,base_relationsCount,kg_degree
0,/home/luis/Documents/FGV/Laboratory/document-g...,"BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1...",6d843a956f1adcbf7394d27457e8342f020a9ed8,121,121
1,/home/luis/Documents/FGV/Laboratory/document-g...,"BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1...",c3dcf35efbbd129ee4e08d34cb26109be2dc9737,36,31
2,/home/luis/Documents/FGV/Laboratory/document-g...,"BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1...",102fafa0ede1ad48835bc9d844bf234a120a5986,35,31
3,/home/luis/Documents/FGV/Laboratory/document-g...,"BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1...",27aa2a37e94374ded441a3fc6491ab467ffa30d1,34,31
4,/home/luis/Documents/FGV/Laboratory/document-g...,"BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1...",c0ca305f958217b50991bc9690c0c2e5abb4a281,27,27
5,/home/luis/Documents/FGV/Laboratory/document-g...,BLUEFLYINC_03_27_2002-EX-10.27-e-business Host...,5b869cb5dd4181e2d1fa9369658bade12df646aa,23,23
6,/home/luis/Documents/FGV/Laboratory/document-g...,BLUEFLYINC_03_27_2002-EX-10.27-e-business Host...,73d6b2e02e341ef55b895737ad0340470d984fb0,8,8
7,/home/luis/Documents/FGV/Laboratory/document-g...,BLUEFLYINC_03_27_2002-EX-10.27-e-business Host...,c222917d975e61e910f6bd65ca4d075e445d9692,8,8
8,/home/luis/Documents/FGV/Laboratory/document-g...,BLUEFLYINC_03_27_2002-EX-10.27-e-business Host...,e0633e8c9bba5eb0a6944c50bb5eb38b71b7b184,6,6
9,/home/luis/Documents/FGV/Laboratory/document-g...,BLUEFLYINC_03_27_2002-EX-10.27-e-business Host...,e66ef6eee17776881f3fa30b0de56fd46c847b2b,6,5


Received notification from DBMS server: <GqlStatusObject gql_status='01N02', status_description='warn: feature deprecated without replacement. id is deprecated and will be removed without a replacement.', position=<SummaryInputPosition line=4, column=49, offset=170>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 170, 'line': 4, 'column': 49}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n    MATCH (ct:Contract)-[:CONTAINS]->(c:Clause)\n    OPTIONAL MATCH (ct)-[:CONTAINS]->(c)-[r]-(:KGNode)<-[:CONTAINS]-(ct)\n    WITH ct.path AS contract, c, count(DISTINCT id(r)) AS kg_degree\n    WHERE kg_degree > 0\n      AND ANY(doc IN $document_list WHERE contract CONTAINS doc)\n    RETURN\n      contract,\n      c.node_id AS clause_no

[SEED]
contract: /home/luis/Documents/FGV/Laboratory/document-graph/infra/json/graph/BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement.json
clause_node_id: 4a2187cb60da96c331433608de54893734fbde45:Clause:BELLICUMPHARMACEUTICALS_INC_05_07_2019-EX-10.1-Supply_Agreement-p-545
source_paragraph_id: BELLICUMPHARMACEUTICALS,INC_05_07_2019-EX-10.1-Supply Agreement-p-545
paragraph_uid: 6d843a956f1adcbf7394d27457e8342f020a9ed8
seed_text: 15.7 Survival. Other than obligations which have accrued and are outstanding as of the date of any expiration or termination of this Agreement, and except as otherwise expressly provided in this Agreement or the Quality Agreement or as otherwise mutually agreed by the Parties in writing, all rights granted and obligations undertaken by the Parties hereunder shall terminate immediately upon the termination or expiration of this Agreement, subject to Section 15.4 above and except for the following

[NODES_2HOP]
- 4a2187cb60da96c331433608de54893734fbd